![header](content/img/header.PNG)

# Analysing GloFAS ensemble forecasts

In this notebook, we continue the Morocco January-April 2026 wet-period application.  
Notebook 4 used the **control forecast**. Here we use the **perturbed ensemble forecasts** to move from a deterministic signal to a probabilistic interpretation.

## Learning objectives

By the end of this notebook, you will be able to:

- open a sequence of GloFAS perturbed forecast files;
- identify the ensemble member dimension;
- compare ensemble forecasts with daily historical reference conditions;
- plot an ensemble spaghetti hydrograph for one forecast issue date;
- summarise ensemble spread using box plots;
- calculate the probability of exceeding the historical daily Q90 threshold;
- map where the ensemble repeatedly indicates unusually high discharge;
- understand how this workflow can be extended to flood return-period thresholds.


## 1. Scientific question

Forecasts are uncertain. A single control forecast can indicate a possible high-flow signal, but it does not tell us how strongly the forecast system supports that signal.

The ensemble forecast gives several possible future trajectories. In this notebook, we ask:

> **How many ensemble members indicated unusually high river discharge during the January-April 2026 wet period in Morocco?**

The main threshold used here is the **daily Q90**, calculated from historical daily discharge for the same day of the year. Values above Q90 are high compared with the historical behaviour of that calendar day.


## 2. Configuration

This notebook reuses the same study basin and historical daily reference period used in Notebook 4.

The analysis focuses on a local domain around the selected basin to keep calculations fast and suitable


In [2]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

warnings.filterwarnings("ignore", message="All-NaN slice encountered")

DATA_DIR = Path("/perm/ecm3644/ecmwf-training/ICAR2026/2026-icar-glofas-training/Data/")

# Historical daily discharge from Part 3
HIST_DAILY_FILE = DATA_DIR / "cems-glofas-historical_morocco_1979_2026.nc"

# Folder containing the perturbed forecast NetCDF files
# Example filename:
# cems-glofas-forecast_2026_01_01_morocco_perturbed.nc
PERTURBED_DIR = DATA_DIR / "forecast/perturbed"
PERTURBED_PATTERN = "cems-glofas-forecast_2026_*_morocco_perturbed*.nc"

# Study basin
LAT_BASIN = 35.2327
LON_BASIN = -6.1209 + 360

# Local analysis domain around the basin
LAT_MIN = 34.50
LAT_MAX = 36.00
LON_MIN = 352.80
LON_MAX = 354.80

# Historical reference period
REF_START = "1979-01-01"
REF_END = "2025-12-31"

# Application period
APP_START = "2026-01-01"
APP_END = "2026-04-30"

print("Historical daily file:", HIST_DAILY_FILE)
print("Perturbed forecast folder:", PERTURBED_DIR)


Historical daily file: /perm/ecm3644/ecmwf-training/ICAR2026/2026-icar-glofas-training/Data/cems-glofas-historical_morocco_1979_2026.nc
Perturbed forecast folder: /perm/ecm3644/ecmwf-training/ICAR2026/2026-icar-glofas-training/Data/forecast/perturbed


## 3. Load the historical daily reference

The ensemble forecasts are daily values, so they must be compared with **daily** historical reference conditions.

For each day of the year, we calculate:

- the historical daily mean;
- Q10;
- Q90.

The local spatial subset is applied before calculating percentiles. This keeps the notebook responsive.


In [3]:
# Open daily historical discharge.
ds_daily = xr.open_dataset(HIST_DAILY_FILE)
dis_daily = ds_daily["dis24"]

# Select the local domain before calculating reference statistics.
dis_daily_area = dis_daily.sel(
    latitude=slice(LAT_MAX, LAT_MIN),
    longitude=slice(LON_MIN, LON_MAX),
)

# Historical discharge during the application period.
hist_2026_daily_area = dis_daily_area.sel(
    valid_time=slice(APP_START, APP_END)
)

# Reference period used to build daily climatological statistics.
reference_daily_area = dis_daily_area.sel(
    valid_time=slice(REF_START, REF_END)
)

# Load the local domain into memory once.
# This is much smaller than the full Morocco grid.
reference_daily_area = reference_daily_area.load()

# Daily reference statistics by day of year.
daily_mean_area = reference_daily_area.groupby("valid_time.dayofyear").mean(
    dim="valid_time",
    skipna=True,
)

daily_q10_area = reference_daily_area.groupby("valid_time.dayofyear").quantile(
    0.10,
    dim="valid_time",
    skipna=True,
)

daily_q90_area = reference_daily_area.groupby("valid_time.dayofyear").quantile(
    0.90,
    dim="valid_time",
    skipna=True,
)

print("Daily historical subset:")
print(dis_daily_area)
print("Daily reference mean:")
print(daily_mean_area)
print("Daily Q10 reference:")
print(daily_q10_area)
print("Daily Q90 reference:")
print(daily_q90_area)


Daily historical subset:
<xarray.DataArray 'dis24' (valid_time: 17335, latitude: 30, longitude: 40)> Size: 83MB
[20802000 values with dtype=float32]
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 139kB 1979-01-02 ... 2026-06-18
  * latitude    (latitude) float64 240B 35.98 35.93 35.88 ... 34.63 34.58 34.53
  * longitude   (longitude) float64 320B 352.8 352.9 352.9 ... 354.7 354.7 354.8
    surface     float64 8B 0.0
Attributes: (12/32)
    GRIB_paramId:                             240024
    GRIB_dataType:                            sfo
    GRIB_numberOfPoints:                      106760
    GRIB_typeOfLevel:                         surface
    GRIB_stepUnits:                           1
    GRIB_stepType:                            avg
    ...                                       ...
    GRIB_name:                                Mean discharge in the last 24 h...
    GRIB_shortName:                           dis24
    GRIB_units:                               m**3 s**-1
  

## 4. Load the perturbed forecasts

Each daily file contains one forecast issue date and several lead times.  
The perturbed forecast also contains an ensemble member dimension, usually named `number`.

The files are opened together into one dataset with:

- `forecast_reference_time`: the date the forecast was issued;
- `forecast_period`: the lead time;
- `valid_time`: the calendar date represented by the forecast;
- `number`: ensemble member.

The files are already downloaded and combined into one file under : Data/glofas_ensemble_forecast_morocco_jan_apr2026.nc


In [7]:
FORECAST_FILE = DATA_DIR / "glofas_ensemble_forecast_morocco_jan_apr2026.nc"

ds_ens = xr.open_dataset(FORECAST_FILE)

ens_area = ds_ens["dis24"]

print(ds_ens)

<xarray.Dataset> Size: 864MB
Dimensions:                  (number: 50, forecast_period: 30,
                              forecast_reference_time: 120, latitude: 30,
                              longitude: 40)
Coordinates:
  * number                   (number) int64 400B 1 2 3 4 5 6 ... 46 47 48 49 50
  * forecast_period          (forecast_period) timedelta64[ns] 240B 1 days .....
    valid_time               (forecast_reference_time, forecast_period) datetime64[ns] 29kB ...
  * forecast_reference_time  (forecast_reference_time) datetime64[ns] 960B 20...
  * latitude                 (latitude) float64 240B 35.98 35.93 ... 34.58 34.53
  * longitude                (longitude) float64 320B 352.8 352.9 ... 354.8
Data variables:
    dis24                    (number, forecast_period, forecast_reference_time, latitude, longitude) float32 864MB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather

## 5. Compare the ensemble with the historical daily reference

For every forecast value, we select the historical reference corresponding to the same day of the year.

This gives two ensemble diagnostics:

1. **daily anomaly**: ensemble forecast minus the historical daily mean;
2. **daily Q90 exceedance**: ensemble forecast above the historical daily Q90 threshold.

The exceedance will later be averaged across ensemble members to produce probabilities.


In [ ]:
# Align historical references to the forecast grid.
daily_mean_on_fc_grid = daily_mean_area.sel(
    latitude=ds_ens.latitude,
    longitude=ds_ens.longitude,
    method="nearest",
)

daily_q90_on_fc_grid = daily_q90_area.sel(
    latitude=ds_ens.latitude,
    longitude=ds_ens.longitude,
    method="nearest",
)

# Select the daily reference matching each forecast valid date.
valid_doy = ds_ens.valid_time.dt.dayofyear

mean_for_ens = daily_mean_on_fc_grid.sel(dayofyear=valid_doy)
q90_for_ens = daily_q90_on_fc_grid.sel(dayofyear=valid_doy)

ensemble_anomaly = ens_area - mean_for_ens
ensemble_above_q90 = ens_area > q90_for_ens

# Align historical references to the forecast grid.
daily_mean_on_fc_grid = daily_mean_area.sel(
    latitude=ds_ens.latitude,
    longitude=ds_ens.longitude,
    method="nearest",
)

daily_q90_on_fc_grid = daily_q90_area.sel(
    latitude=ds_ens.latitude,
    longitude=ds_ens.longitude,
    method="nearest",
)

# Select the daily reference matching each forecast valid date.
valid_doy = ds_ens.valid_time.dt.dayofyear

mean_for_ens = daily_mean_on_fc_grid.sel(dayofyear=valid_doy)
q90_for_ens = daily_q90_on_fc_grid.sel(dayofyear=valid_doy)

ensemble_anomaly = ens_area - mean_for_ens
ensemble_above_q90 = ens_area > q90_for_ens

# Select one forecast issue date
ISSUE_DATE = np.datetime64("2026-02-03")

ds_one = ds_ens.sel(
    forecast_reference_time=ISSUE_DATE,
    method="nearest",
)

ens_one_point = ds_one["dis24"].sel(
    latitude=LAT_BASIN,
    longitude=LON_BASIN,
    method="nearest",
).compute()

valid_time_one = ds_one["valid_time"].compute()
doy_one = valid_time_one.dt.dayofyear

mean_point_one = daily_mean_area.sel(
    latitude=LAT_BASIN,
    longitude=LON_BASIN,
    method="nearest",
).sel(dayofyear=doy_one).compute()

q10_point_one = daily_q10_area.sel(
    latitude=LAT_BASIN,
    longitude=LON_BASIN,
    method="nearest",
).sel(dayofyear=doy_one).compute()

q90_point_one = daily_q90_area.sel(
    latitude=LAT_BASIN,
    longitude=LON_BASIN,
    method="nearest",
).sel(dayofyear=doy_one).compute()

# Probability of exceeding Q90 at the basin
q90_probability_one = (
    (ens_one_point > q90_point_one).mean(dim="number") * 100
).compute()

# Regional probability map: first 10 lead days
ens_above_q90_one = ensemble_above_q90.sel(
    forecast_reference_time=ds_one.forecast_reference_time
)

q90_probability_map = (
    ens_above_q90_one
    .isel(forecast_period=slice(0, 10))
    .mean(dim=("number", "forecast_period"), skipna=True)
    * 100
).compute()

print("Diagnostics ready for plotting.")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

ax = axes[0, 0]

for member in ens_one_point["number"].values:
    ax.plot(
        valid_time_one.values,
        ens_one_point.sel(number=member).values,
        color="grey",
        alpha=0.25,
        linewidth=0.8,
    )

ens_median = ens_one_point.median(dim="number")

ax.plot(
    valid_time_one.values,
    ens_median.values,
    color="blue",
    linewidth=2.5,
    label="Ensemble median",
)

ax.plot(
    valid_time_one.values,
    mean_point_one.values,
    color="black",
    linestyle="--",
    linewidth=1.5,
    label="Historical daily mean",
)

ax.fill_between(
    valid_time_one.values,
    q10_point_one.values,
    q90_point_one.values,
    color="lightgrey",
    alpha=0.6,
    label="Historical Q10–Q90 range",
)

ax.plot(
    valid_time_one.values,
    q90_point_one.values,
    color="red",
    linestyle="--",
    linewidth=1.5,
    label="Historical daily Q90",
)

ax.set_title("a) Ensemble spaghetti plot at the study basin")
ax.set_ylabel("River discharge (m³ s⁻¹)")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)

ax = axes[0, 1]

df_box = ens_one_point.to_pandas().T
df_box.index = pd.to_datetime(valid_time_one.values)

ax.boxplot(
    [df_box.loc[date].dropna().values for date in df_box.index],
    positions=np.arange(len(df_box.index)),
    widths=0.6,
    showfliers=False,
)

ax.plot(
    np.arange(len(df_box.index)),
    q90_point_one.values,
    color="red",
    linestyle="--",
    linewidth=1.5,
    label="Historical daily Q90",
)

ax.plot(
    np.arange(len(df_box.index)),
    mean_point_one.values,
    color="black",
    linestyle="--",
    linewidth=1.2,
    label="Historical daily mean",
)

ax.set_xticks(np.arange(len(df_box.index))[::5])
ax.set_xticklabels(
    [d.strftime("%d %b") for d in df_box.index[::5]],
    rotation=45,
    ha="right",
)

ax.set_title("b) Ensemble spread by valid date")
ax.set_ylabel("River discharge (m³ s⁻¹)")
ax.grid(True, axis="y", alpha=0.3)
ax.legend(fontsize=8)

ax = axes[1, 0]

ax.plot(
    valid_time_one.values,
    q90_probability_one.values,
    color="blue",
    marker="o",
    linewidth=2,
)

ax.axhline(75, color="red", linestyle="--", linewidth=1, label="High ≥ 75%")
ax.axhline(50, color="orange", linestyle="--", linewidth=1, label="Moderate ≥ 50%")
ax.axhline(25, color="green", linestyle="--", linewidth=1, label="Low ≥ 25%")

ax.set_title("c) Probability of exceeding daily Q90 at the basin")
ax.set_ylabel("Members above Q90 (%)")
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)

ax = axes[1, 1]

q90_probability_map.plot(
    ax=ax,
    x="longitude",
    y="latitude",
    cmap="YlOrRd",
    vmin=0,
    vmax=100,
    cbar_kwargs={"label": "Members above daily Q90 (%)"},
)

ax.scatter(
    LON_BASIN,
    LAT_BASIN,
    marker="x",
    s=80,
    color="black",
    label="Study basin",
)

ax.set_title("d) Q90 exceedance probability map\nIssued 15 Jan, lead days 1–10")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend(fontsize=8)

for ax in axes.ravel():
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## 6. Inspect one ensemble forecast cycle

We start with a single forecast issue date.  
This is the clearest way to understand what the ensemble is telling us.

The spaghetti plot shows each ensemble member as one possible future trajectory.  
The Q10-Q90 band shows the central historical daily range.  
When many ensemble members rise above Q90, the forecast indicates increased confidence in unusually high discharge.


In [ ]:
ISSUE_DATE = np.datetime64("2026-02-03")

ds_one = ds_ens.sel(
    forecast_reference_time=ISSUE_DATE,
    method="nearest",
)

ens_one_point = ds_one["dis24"].sel(
    latitude=LAT_BASIN,
    longitude=LON_BASIN,
    method="nearest",
).compute()

valid_time_one = ds_one["valid_time"].compute()
doy_one = valid_time_one.dt.dayofyear

mean_point_one = daily_mean_area.sel(
    latitude=LAT_BASIN,
    longitude=LON_BASIN,
    method="nearest",
).sel(dayofyear=doy_one).compute()

q10_point_one = daily_q10_area.sel(
    latitude=LAT_BASIN,
    longitude=LON_BASIN,
    method="nearest",
).sel(dayofyear=doy_one).compute()

q90_point_one = daily_q90_area.sel(
    latitude=LAT_BASIN,
    longitude=LON_BASIN,
    method="nearest",
).sel(dayofyear=doy_one).compute()

print("Selected issue date:", pd.to_datetime(ds_one.forecast_reference_time.values).date())
print(ens_one_point)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

# Spaghetti: one line per ensemble member.
for member in ens_one_point["number"].values:
    ax.plot(
        valid_time_one.values,
        ens_one_point.sel(number=member).values,
        color="tab:blue",
        alpha=0.25,
        linewidth=1,
    )

# Ensemble median.
ens_median = ens_one_point.median(dim="number")
ax.plot(
    valid_time_one.values,
    ens_median.values,
    color="tab:blue",
    linewidth=2.5,
    label="Ensemble median",
)

# Historical daily reference.
ax.plot(
    valid_time_one.values,
    mean_point_one.values,
    color="black",
    linestyle="--",
    linewidth=1.5,
    label="Historical daily mean",
)

ax.fill_between(
    valid_time_one.values,
    q10_point_one.values,
    q90_point_one.values,
    color="lightgrey",
    alpha=0.6,
    label="Historical Q10-Q90 range",
)

ax.plot(
    valid_time_one.values,
    q90_point_one.values,
    color="red",
    linestyle="--",
    linewidth=1.5,
    label="Historical daily Q90",
)

ax.set_title("GloFAS perturbed forecast spaghetti plot at the study basin")
ax.set_xlabel("Forecast valid date")
ax.set_ylabel("River discharge (m³ s⁻¹)")
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()


## 7. Box plot of ensemble spread

The spaghetti plot shows individual members.  
The box plot summarises the ensemble distribution for each lead day.

This helps identify whether the ensemble signal is narrow and confident, or wide and uncertain.


In [ ]:
# Convert one issue-date ensemble forecast to a tidy dataframe.
df_box = ens_one_point.to_pandas().T
df_box.index = pd.to_datetime(valid_time_one.values)

fig, ax = plt.subplots(figsize=(13, 5))

ax.boxplot(
    [df_box.loc[date].dropna().values for date in df_box.index],
    positions=np.arange(len(df_box.index)),
    widths=0.6,
    showfliers=False,
)

ax.plot(
    np.arange(len(df_box.index)),
    q90_point_one.values,
    color="red",
    linestyle="--",
    linewidth=1.5,
    label="Historical daily Q90",
)

ax.plot(
    np.arange(len(df_box.index)),
    mean_point_one.values,
    color="black",
    linestyle="--",
    linewidth=1.2,
    label="Historical daily mean",
)

ax.set_xticks(np.arange(len(df_box.index))[::3])
ax.set_xticklabels(
    [d.strftime("%d %b") for d in df_box.index[::3]],
    rotation=45,
    ha="right",
)

ax.set_title("Ensemble spread by valid date")
ax.set_xlabel("Forecast valid date")
ax.set_ylabel("River discharge (m³ s⁻¹)")
ax.grid(True, axis="y", alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()


## 8. Probability of exceeding daily Q90 at the basin

For the ensemble, exceedance probability is calculated as:

> percentage of ensemble members above the historical daily Q90 threshold.

This is the key difference from Notebook 4.  
The control forecast answered whether one trajectory exceeded Q90.  
The ensemble forecast tells us how many members support that exceedance.


In [ ]:
# Basin point ensemble exceedance for the selected issue date.
q90_exceedance_one = ens_one_point > q90_point_one

q90_probability_one = (
    q90_exceedance_one.mean(dim="number") * 100
).compute()

fig, ax = plt.subplots(figsize=(11, 4))

ax.plot(
    valid_time_one.values,
    q90_probability_one.values,
    marker="o",
    linewidth=2,
)

ax.axhline(50, color="grey", linestyle="--", linewidth=1, label="50% of members")

ax.set_title("Probability of exceeding daily Q90 at the study basin")
ax.set_xlabel("Forecast valid date")
ax.set_ylabel("Members above Q90 (%)")
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()


## 9. Regional probability map

We now move from one basin point to the local domain.

For each grid cell, we calculate the percentage of ensemble members exceeding daily Q90.  
This can be done for one forecast issue date and one lead time, or averaged over a lead-time window.

Here we calculate the average Q90 exceedance probability over the first 10 forecast days for the selected forecast issue date.


In [ ]:
LEAD_START = 0
LEAD_END = 10

ens_above_q90_one = ensemble_above_q90.sel(
    forecast_reference_time=ds_one.forecast_reference_time
)

q90_probability_map = (
    ens_above_q90_one
    .isel(forecast_period=slice(LEAD_START, LEAD_END))
    .mean(dim=("number", "forecast_period"), skipna=True)
    * 100
).compute()

print(q90_probability_map)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

q90_probability_map.plot(
    ax=ax,
    x="longitude",
    y="latitude",
    cmap="YlOrRd",
    vmin=0,
    vmax=100,
    cbar_kwargs={"label": "Members above daily Q90 (%)"},
)

ax.scatter(
    LON_BASIN,
    LAT_BASIN,
    marker="x",
    s=80,
    color="black",
    label="Study basin",
)

issue_date_label = pd.to_datetime(ds_one.forecast_reference_time.values).strftime("%Y-%m-%d")
ax.set_title(f"Ensemble Q90 exceedance probability\nIssued {issue_date_label}, lead days 1-10")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()

plt.tight_layout()
plt.show()


## 10. Forecast evolution across issue dates

Operationally, forecasters look for persistence across successive forecasts.

Here we compare several issue dates at the basin point.  
If successive forecasts show a similar high-flow signal, confidence in the forecast message increases.


In [ ]:
hist_2026_point = (
    hist_2026_daily_area
    .sel(
        latitude=LAT_BASIN,
        longitude=LON_BASIN,
        method="nearest",
    )
    .compute()
)

hist_2026_point = hist_2026_daily_area.sel(
    latitude=LAT_BASIN,
    longitude=LON_BASIN,
    method="nearest",
).compute()

print(hist_2026_point)
SELECTED_ISSUE_DATES = [
    "2026-01-01",
    "2026-01-15",
    "2026-02-01",
    "2026-02-15",
]

fig, ax = plt.subplots(figsize=(13, 5))

for issue_date in SELECTED_ISSUE_DATES:
    ds_sel = ds_ens.sel(
        forecast_reference_time=np.datetime64(issue_date),
        method="nearest",
    )

    ens_sel_point = ds_sel["dis24"].sel(
        latitude=LAT_BASIN,
        longitude=LON_BASIN,
        method="nearest",
    ).compute()

    valid_time_sel = ds_sel["valid_time"].compute()
    ens_median_sel = ens_sel_point.median(dim="number")

    ax.plot(
        valid_time_sel.values,
        ens_median_sel.values,
        linewidth=2,
        label=f"Median issued {issue_date}",
    )

# Add historical 2026 daily discharge.
ax.plot(
    hist_2026_point["valid_time"].values,
    hist_2026_point.values,
    color="black",
    linewidth=2.5,
    label="Historical daily 2026",
)

# Daily historical reference for the event period.
event_doy = hist_2026_point["valid_time"].dt.dayofyear

mean_event = daily_mean_area.sel(
    latitude=LAT_BASIN,
    longitude=LON_BASIN,
    method="nearest",
).sel(dayofyear=event_doy).compute()

q10_event = daily_q10_area.sel(
    latitude=LAT_BASIN,
    longitude=LON_BASIN,
    method="nearest",
).sel(dayofyear=event_doy).compute()

q90_event = daily_q90_area.sel(
    latitude=LAT_BASIN,
    longitude=LON_BASIN,
    method="nearest",
).sel(dayofyear=event_doy).compute()

ax.plot(
    hist_2026_point["valid_time"].values,
    mean_event.values,
    color="grey",
    linestyle="--",
    linewidth=1.5,
    label="Historical daily mean",
)

ax.fill_between(
    hist_2026_point["valid_time"].values,
    q10_event.values,
    q90_event.values,
    color="lightgrey",
    alpha=0.5,
    label="Historical Q10-Q90 range",
)

ax.set_title("Evolution of ensemble-median forecasts at the study basin")
ax.set_xlabel("Valid date")
ax.set_ylabel("River discharge (m³ s⁻¹)")
ax.grid(True, alpha=0.3)
ax.legend(ncol=2)

ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))

plt.tight_layout()
plt.show()


## 11. Flood return-period thresholds

The daily Q90 threshold identifies unusually high discharge relative to the historical distribution.  
Flood return-period thresholds provide a stronger flood-oriented interpretation.

If return-period threshold files are available, the ensemble probability can be calculated in the same way:

```python
probability = (ensemble_discharge > flood_threshold).mean(dim="number") * 100
```


In [ ]:
# Import flood thresholds

THRESHOLD_DIR = Path("/perm/ecm3644/glofas/Static_maps/")

FLOOD_THRESHOLD_FILES = {
    "RP2": THRESHOLD_DIR / "flood_threshold_glofas_v4_rl_2.nc",
    "RP5": THRESHOLD_DIR / "flood_threshold_glofas_v4_rl_5.nc",
    "RP20": THRESHOLD_DIR / "flood_threshold_glofas_v4_rl_20.nc",
}

print("Optional flood threshold files:")
for name, path in FLOOD_THRESHOLD_FILES.items():
    print(name, path)


In [ ]:
ISSUE_DATE = np.datetime64("2026-02-03")

THRESHOLD_DIR = Path("/perm/ecm3644/glofas/Static_maps/")

FLOOD_THRESHOLD_FILES = {
    "RP2": THRESHOLD_DIR / "flood_threshold_glofas_v4_rl_2.nc",
    "RP5": THRESHOLD_DIR / "flood_threshold_glofas_v4_rl_5.nc",
    "RP20": THRESHOLD_DIR / "flood_threshold_glofas_v4_rl_20.nc",
}

def open_threshold_point(path, lat_value, lon_value):
    ds_thr = xr.open_dataset(path)

    # Use first data variable, unless the variable name is known.
    var = list(ds_thr.data_vars)[0]

    # Threshold files use lat/lon, while forecast files use latitude/longitude.
    lat_name = "latitude" if "latitude" in ds_thr.coords else "lat"
    lon_name = "longitude" if "longitude" in ds_thr.coords else "lon"

    # Convert longitude from 0–360 to -180–180 if needed.
    lon_coord = ds_thr[lon_name]
    if float(lon_coord.max()) <= 180 and lon_value > 180:
        lon_select = lon_value - 360
    else:
        lon_select = lon_value

    thr_point = ds_thr[var].sel(
        {
            lat_name: lat_value,
            lon_name: lon_select,
        },
        method="nearest",
    ).item()

    return float(thr_point)


rp2 = open_threshold_point(FLOOD_THRESHOLD_FILES["RP2"], LAT_BASIN, LON_BASIN)
rp5 = open_threshold_point(FLOOD_THRESHOLD_FILES["RP5"], LAT_BASIN, LON_BASIN)
rp20 = open_threshold_point(FLOOD_THRESHOLD_FILES["RP20"], LAT_BASIN, LON_BASIN)

print("Official GloFAS thresholds at the basin point:")
print(f"RP2 :  {rp2:.2f} m³/s")
print(f"RP5 :  {rp5:.2f} m³/s")
print(f"RP20: {rp20:.2f} m³/s")

ds_issue = ds_ens.sel(
    forecast_reference_time=ISSUE_DATE,
    method="nearest",
)

ens_issue_point = (
    ds_issue["dis24"]
    .sel(latitude=LAT_BASIN, longitude=LON_BASIN, method="nearest")
    .compute()
)

valid_time_issue = ds_issue["valid_time"].compute()

ens_median = ens_issue_point.median(dim="number")
ens_q25 = ens_issue_point.quantile(0.25, dim="number")
ens_q75 = ens_issue_point.quantile(0.75, dim="number")
ens_min = ens_issue_point.min(dim="number")
ens_max = ens_issue_point.max(dim="number")

ymax = max(
    float(ens_max.max()),
    rp20 * 1.2,
)

fig, ax = plt.subplots(figsize=(14, 7))

ax.fill_between(
    valid_time_issue.values,
    0,
    rp2,
    alpha=0.12,
    label="Below RP2",
)

ax.fill_between(
    valid_time_issue.values,
    rp2,
    rp5,
    alpha=0.12,
    label="RP2–RP5",
)

ax.fill_between(
    valid_time_issue.values,
    rp5,
    rp20,
    alpha=0.12,
    label="RP5–RP20",
)

ax.fill_between(
    valid_time_issue.values,
    rp20,
    ymax,
    alpha=0.12,
    label="Above RP20",
)

for member in ens_issue_point["number"].values:
    ax.plot(
        valid_time_issue.values,
        ens_issue_point.sel(number=member).values,
        color="grey",
        alpha=0.25,
        linewidth=0.8,
    )

ax.fill_between(
    valid_time_issue.values,
    ens_q25.values,
    ens_q75.values,
    alpha=0.35,
    label="Ensemble interquartile range",
)

ax.plot(
    valid_time_issue.values,
    ens_median.values,
    linewidth=2.5,
    label="Ensemble median",
)

ax.axhline(rp2, linestyle="--", linewidth=1.5, label=f"RP2: {rp2:.1f} m³/s")
ax.axhline(rp5, linestyle="--", linewidth=1.5, label=f"RP5: {rp5:.1f} m³/s")
ax.axhline(rp20, linestyle="--", linewidth=1.5, label=f"RP20: {rp20:.1f} m³/s")

issue_label = pd.to_datetime(ds_issue.forecast_reference_time.values).strftime("%Y-%m-%d")

ax.set_title(f"GloFAS ensemble forecast issued {issue_label} with official flood thresholds")
ax.set_xlabel("Forecast valid date")
ax.set_ylabel("River discharge (m³ s⁻¹)")
ax.set_ylim(0, ymax)
ax.grid(True, alpha=0.3)
ax.legend(ncol=2, fontsize=9)

plt.tight_layout()
plt.show()

## 12. Exercise

Choose another basin point from the station-mapping notebook and repeat the ensemble analysis.

1. Extract the ensemble forecast at the new point.
2. Plot one spaghetti hydrograph.
3. Add the historical daily Q10-Q90 range.
4. Calculate the probability of exceeding daily Q90.
5. Compare the signal with the study basin used in this notebook.

